# MINI SHAZAM

## BIBLIOTEKE

In [1]:
import os
from pathlib import Path
import numpy as np
import random 

ROOT_DIR = str(Path('..').parent.resolve())
SONG_DIR = os.path.join(ROOT_DIR, 'songs')
NOISE_DIR = os.path.join(ROOT_DIR,'noise')

In [2]:
from pydub import AudioSegment
from pydub.effects import normalize
from pydub.generators import WhiteNoise

## PREPROCESSING

In [3]:
def add_background_noise(clean_audio: AudioSegment, noise_file_path: str, noise_reduction_db: int = 15) -> AudioSegment:
    """
    Takes a clean AudioSegment and overlays background noise from a file.
    Automatically loops or trims the noise to match the clean audio's length.
    """
    # 1. Load the background noise
    # Using from_file is safer than from_mp3 as it handles .wav and .mp3 seamlessly
    noise_audio = AudioSegment.from_file(noise_file_path)
    
    # 2. If the noise is shorter than the clean audio, loop it
    if len(noise_audio) < len(clean_audio):
        loop_count = (len(clean_audio) // len(noise_audio)) + 1
        noise_audio = noise_audio * loop_count
        
    # 3. Trim the noise to be the EXACT same length as the clean audio
    noise_audio = noise_audio[:len(clean_audio)]
    
    # 4. Lower the volume of the background noise so it doesn't overpower the main track
    noise_audio = noise_audio - noise_reduction_db
    
    # 5. Overlay and return
    return normalize(clean_audio.overlay(noise_audio))

In [4]:
def take_one_for_dataset(song_name, effects_list):
    # 1. Clean the filename (removes '.mp3' so your files look like 'track1_chunk_0.0s.wav')
    clean_name = Path(song_name).stem

    # 2. Load your preprocessed audio
    audio = AudioSegment.from_mp3(os.path.join(SONG_DIR, song_name))
    audio = audio.set_channels(1).set_frame_rate(16000)
    audio = normalize(audio)
    
    # 3. Define your window and step size
    window_size = 5000  # 5 seconds
    step_size = 2500    # 2.5 seconds overlap

    # 4. Loop through the audio and slice it
    for i in range(0, len(audio) - window_size, step_size):
        original_chunk = audio[i : i + window_size]
        
        # Choose a random noise file path from the list passed into the function
        effect_path = random.choice(effects_list)
        
        # FIX: Set reduction to 15! If it is 0, the noise will drown out the song.
        noice_reduction_db = random.randint(-10,5)
        noisy_chunk = add_background_noise(original_chunk, effect_path, noise_reduction_db=noice_reduction_db)
        
        # Create a clean unique filename
        start_time_sec = i / 1000
        base_filename = f"{clean_name}_chunk_{start_time_sec}s.wav"
        
        # FIX: Use os.path.join for safe directory paths
        original_filename = os.path.join(ORIGINAL_OUT_DIR, base_filename)
        noisy_filename = os.path.join(NOISY_OUT_DIR, base_filename)
        
        # Export the chunks
        original_chunk.export(original_filename, format="wav")
        noisy_chunk.export(noisy_filename, format="wav")
        
    print(f"Sequential slicing complete for: {clean_name}")

In [5]:
effects = [os.path.join(NOISE_DIR,effect) for effect in os.listdir(NOISE_DIR)]
# Define your directories (Assuming ROOT_DIR, SONG_DIR, and NOISE_DIR are defined above)
ORIGINAL_OUT_DIR = os.path.join(ROOT_DIR, "original_dataset")
NOISY_OUT_DIR = os.path.join(ROOT_DIR, "noisy_dataset")

# 1. NEW: Check and create the export folders if they don't exist
os.makedirs(ORIGINAL_OUT_DIR, exist_ok=True)
os.makedirs(NOISY_OUT_DIR, exist_ok=True)
print("Output directories verified/created.")
for song_name in os.listdir(SONG_DIR):
    take_one_for_dataset(song_name,effects)
    

Output directories verified/created.
Sequential slicing complete for: Borderline
Sequential slicing complete for: Breathe-deeper
Sequential slicing complete for: Grimmer
Sequential slicing complete for: Instant-Destiny
Sequential slicing complete for: Is-it-true
Sequential slicing complete for: It-might-be-time
Sequential slicing complete for: Lost-in-yesterday
Sequential slicing complete for: On-track
Sequential slicing complete for: One-more-hour
Sequential slicing complete for: One-more-year
Sequential slicing complete for: Posthumous-forgivness
Sequential slicing complete for: Tomorrow's-dust


## DATASET

In [6]:
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
import torch

In [7]:
class SimpleAudioDataset(Dataset):
    def __init__(self, data_paths, labels):
        self.data_paths = data_paths
        self.labels = labels

        # Initialize the audio transforms
        self.mel_transform = T.MelSpectrogram(
            sample_rate=16000, n_fft=1024, hop_length=512, n_mels=64
        )
        self.amp_to_db = T.AmplitudeToDB()
        
        # This is exactly 5 seconds of audio at these settings
        self.target_time_frames = 157 

    def __len__(self):
        return len(self.data_paths)

    def __getitem__(self, index):
        file_path = self.data_paths[index]
        label = self.labels[index]

        # 1. Load the audio file
        waveform, _ = torchaudio.load(file_path)
        
        # SAFETY NET 1: Force Mono (1 Channel)
        # If the audio has 2 channels (stereo), take the average to make it mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # 2. Convert to Decibel Mel Spectrogram
        mel_spec = self.mel_transform(waveform)
        db_mel_spec = self.amp_to_db(mel_spec)

        # SAFETY NET 2: Force exact time length (157 frames)
        current_frames = db_mel_spec.shape[2]
        
        if current_frames > self.target_time_frames:
            # It's too long! Crop off the extra frames at the end
            db_mel_spec = db_mel_spec[:, :, :self.target_time_frames]
            
        elif current_frames < self.target_time_frames:
            # It's too short! Pad the end with mathematical silence (0s)
            padding_amount = self.target_time_frames - current_frames
            db_mel_spec = torch.nn.functional.pad(db_mel_spec, (0, padding_amount))

        # Now we absolutely guarantee the shape is [1, 64, 157]
        return db_mel_spec, label

In [8]:
import os
from sklearn.model_selection import train_test_split

# 1. Grab all the filenames (assuming both folders have identically named files)
filenames = sorted(os.listdir(ORIGINAL_OUT_DIR))

# Create our string-to-int map
labels = [name.split('_')[0] for name in filenames]
unique_strings = sorted(set(labels))
string_to_int_map = {label: i for i, label in enumerate(unique_strings)}

# 2. Build dedicated lists for Clean and Noisy
clean_paths = []
clean_labels = []
noisy_paths = []
noisy_labels = []

for song_name in filenames:
    integer_label = string_to_int_map[song_name.split('_')[0]]
    
    clean_paths.append(os.path.join(ORIGINAL_OUT_DIR, song_name))
    clean_labels.append(integer_label)
    
    noisy_paths.append(os.path.join(NOISY_OUT_DIR, song_name))
    noisy_labels.append(integer_label)

# 3. Split ONLY the noisy files!
train_noisy_paths, test_noisy_paths, train_noisy_labels, test_noisy_labels = train_test_split(
    noisy_paths, 
    noisy_labels, 
    test_size=0.2, 
    random_state=42
)

# 4. Combine 100% of Clean Files + 80% of Noisy Files for the Training Dataloader
final_train_paths = clean_paths + train_noisy_paths
final_train_labels = clean_labels + train_noisy_labels

# 5. Keep the 20% Noisy files isolated for the final Accuracy Test
final_test_paths = test_noisy_paths
final_test_labels = test_noisy_labels

print(f"Training on {len(final_train_paths)} total files (Clean + 80% Noisy)")
print(f"Holding back {len(final_test_paths)} unseen Noisy files for testing!")

Training on 2448 total files (Clean + 80% Noisy)
Holding back 272 unseen Noisy files for testing!


In [9]:
train_dataset = SimpleAudioDataset(data_paths=final_train_paths, labels=final_train_labels)

# 2. Create the DataLoader factory line
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=64,       # Number of audio files to process simultaneously
    shuffle=True,        # Mixes up the order every epoch so the model learns features, not order
    drop_last=True       # Trims the last batch if it doesn't perfectly divide by 64
)

## MODEL

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights

class AudioResNet(nn.Module):
    def __init__(self, embedding_size=128):
        super().__init__()
        
        # 1. Load the pre-trained ResNet-18 (Transfer Learning)
        # Using pre-trained weights gives the model a massive head start in recognizing audio textures
        self.backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        
        # 2. MODIFY INPUT: Change the first layer to accept 1-Channel audio instead of 3-Channel RGB
        # We keep the exact same kernel size and stride as the original model
        self.backbone.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=64, 
            kernel_size=(7, 7), 
            stride=(2, 2), 
            padding=(3, 3), 
            bias=False
        )
        
        # 3. MODIFY OUTPUT: Replace the final Fully Connected (fc) layer
        # Standard ResNet outputs 1000 categories. We need it to output our 128-number embedding.
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(num_features, embedding_size)

    def forward(self, x):
        # Pass the 2D spectrogram tensor through the ResNet backbone
        embeddings = self.backbone(x)
        
        # CRITICAL FOR TRIPLET LOSS: L2 Normalize the embeddings
        # This forces all audio clips to sit on the surface of a perfect mathematical sphere,
        # making the distance calculations highly accurate.
        normalized_embeddings = F.normalize(embeddings, p=2, dim=1)
        
        return normalized_embeddings

In [11]:
train_model = AudioResNet(embedding_size=128)

dummy_spectrograms = torch.randn(64, 1, 64, 157)
    
# Run the fake data through the model
output = train_model(dummy_spectrograms)

print("--- Model Architecture Test ---")
print(f"Input Spectrogram Shape:  {dummy_spectrograms.shape}")
print(f"Output Embedding Shape:   {output.shape}")
print("Test Passed: The model successfully output 128 normalized coordinates per audio clip!")

--- Model Architecture Test ---
Input Spectrogram Shape:  torch.Size([64, 1, 64, 157])
Output Embedding Shape:   torch.Size([64, 128])
Test Passed: The model successfully output 128 normalized coordinates per audio clip!


## LOSS AND OPTIMIZER

In [12]:
from pytorch_metric_learning import losses, miners

# The Miner: Set to "hard" so it actively looks for the most difficult negatives to tell apart
miner = miners.TripletMarginMiner(margin=1.0, type_of_triplets="hard")

# The Loss Function
loss_func = losses.TripletMarginLoss(margin=1.0)

In [13]:
import torch.optim as optim

# Move the model to your GPU if you have one
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_model = train_model.to(device)

# 2. Initialize the Adam Optimizer
# lr=1e-4 is the safe, standard learning rate for fine-tuning pre-trained models
optimizer = optim.Adam(train_model.parameters(), lr=1e-4)

## TRAINING

In [15]:
MODEL_DIR = os.path.join(ROOT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

In [27]:


from tqdm import tqdm

num_epochs = 20

for epoch in range(num_epochs):
    print(f"\n=== EPOCH {epoch+1}/{num_epochs} ===")
    
    # BEST PRACTICE: Always put the model in train mode INSIDE the loop
    train_model.train()
    
    # We put the tqdm progress bar here!
    loop = tqdm(train_dataloader)
    
    for batch in loop:
        # 1. Get the normal batch of data
        spectrograms, labels = batch
        spectrograms, labels = spectrograms.to(device), labels.to(device)
        
        optimizer.zero_grad()

        # 2. Pass the entire batch through the CNN to get embeddings
        embeddings = train_model(spectrograms)

        # 3. NINE-LINE MAGIC: The Miner finds the hardest triplets in this batch!
        indices_tuple = miner(embeddings, labels)

        # 4. Calculate the loss using ONLY the hard triplets found by the miner
        loss = loss_func(embeddings, labels, indices_tuple)

        # 5. Backpropagate and update
        loss.backward()
        optimizer.step()

        # Update the progress bar text dynamically so you can watch it drop
        loop.set_postfix(Loss=loss.item(), Triplets=miner.num_triplets)

    # BUG FIX: Save the model into the dedicated MODEL_DIR!
    save_path = os.path.join(MODEL_DIR, f"audio_resnet_epoch_{epoch+1}.pth")
    torch.save(train_model.state_dict(), save_path)


=== EPOCH 1/20 ===


100%|██████████| 42/42 [01:14<00:00,  1.78s/it, Loss=1.05, Triplets=6860] 



=== EPOCH 2/20 ===


100%|██████████| 42/42 [01:13<00:00,  1.76s/it, Loss=1.05, Triplets=4609]



=== EPOCH 3/20 ===


...

100%|██████████| 42/42 [01:15<00:00,  1.79s/it, Loss=1.02, Triplets=123]



=== EPOCH 19/20 ===


100%|██████████| 42/42 [01:15<00:00,  1.80s/it, Loss=1.05, Triplets=456]



=== EPOCH 20/20 ===


100%|██████████| 42/42 [01:14<00:00,  1.78s/it, Loss=1.01, Triplets=6]  


In [16]:
model = AudioResNet(embedding_size=128).to(device)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR,"audio_resnet_epoch_20.pth")))

<All keys matched successfully>

In [17]:
model.eval()

AudioResNet(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05

In [18]:
int_to_string_map = {v: k for k, v in string_to_int_map.items()}

In [19]:
print("Building the database of clean audio fingerprints...")

original_files = os.listdir(ORIGINAL_OUT_DIR)

list_of_originals = [os.path.join(ORIGINAL_OUT_DIR, song_name) for song_name in original_files]
original_labels = [song_name.split('_')[0] for song_name in original_files]

# We will use your DataLoader, but without shuffling, so we keep track of the order!
database_dataset = SimpleAudioDataset(
    data_paths=list_of_originals,
      labels = [string_to_int_map[label] for label in original_labels]
      )
database_dataloader = DataLoader(database_dataset, batch_size=64, shuffle=False)

all_database_embeddings = []
all_database_labels = []

# Turn off gradient math to save massive amounts of RAM
with torch.no_grad():
    for spectrograms, labels in database_dataloader:
        spectrograms = spectrograms.to(device)
        
        # Get the 128-number coordinates
        embeddings = model(spectrograms)
        
        all_database_embeddings.append(embeddings.cpu())
        all_database_labels.extend(labels.tolist())

# Stack the list of batches into one massive matrix
# Shape will be [Total_Clean_Chunks, 128]
database_matrix = torch.cat(all_database_embeddings, dim=0)
print(f"Database built! Stored {database_matrix.shape[0]} unique fingerprints.")

Building the database of clean audio fingerprints...
Database built! Stored 1360 unique fingerprints.


In [20]:
def identify_song(noisy_embedding, database_matrix, database_labels, int_to_string_map):
    """
    Finds the closest song in the database to the noisy query.
    """
    # 1. Calculate the distance between the query and EVERY item in the database instantly
    # torch.cdist calculates the Euclidean Distance between two matrices
    distances = torch.cdist(noisy_embedding, database_matrix)
    
    # 2. Find the index of the absolute smallest distance
    closest_index = torch.argmin(distances).item()
    
    # 3. Look up what song that index belongs to
    predicted_int_label = database_labels[closest_index]
    predicted_song_name = int_to_string_map[predicted_int_label]
    
    return predicted_song_name

In [21]:
# Assuming you have a list of all your NOISY file paths and their true integer labels
# noisy_paths = [...]
# noisy_labels = [...]


noisy_dataset = SimpleAudioDataset(
    data_paths=final_test_paths,
    labels=final_test_labels
    )
noisy_dataloader = DataLoader(noisy_dataset, batch_size=1, shuffle=True)

print("\n--- Starting Accuracy Test ---")
correct_predictions = 0
total_tests = len(noisy_dataset)

with torch.no_grad():
    for i, (noisy_spectrogram, true_label) in enumerate(noisy_dataloader):
        noisy_spectrogram = noisy_spectrogram.to(device)
        
        # 1. Get the coordinate for the noisy audio
        noisy_embedding = model(noisy_spectrogram)
        
        # 2. Search the database for the closest match
        predicted_song = identify_song(
            noisy_embedding.cpu(), 
            database_matrix, 
            all_database_labels, 
            int_to_string_map
        )
        
        # 3. Check if it was right!
        true_song = int_to_string_map[true_label.item()]
        
        if predicted_song == true_song:
            correct_predictions += 1
            
        # Print the first 10 tests just so you can see it working
        if i < 10:
            print(f"Test {i+1}: True Song: {true_song} | Predicted: {predicted_song}")

# Calculate final grade
accuracy = (correct_predictions / total_tests) * 100
print(f"\nFINAL ACCURACY: {accuracy:.2f}% ({correct_predictions}/{total_tests} correct)")


--- Starting Accuracy Test ---
Test 1: True Song: Borderline | Predicted: Borderline
Test 2: True Song: Grimmer | Predicted: One-more-year
Test 3: True Song: Instant-Destiny | Predicted: Instant-Destiny
Test 4: True Song: One-more-year | Predicted: One-more-year
Test 5: True Song: Posthumous-forgivness | Predicted: Posthumous-forgivness
Test 6: True Song: One-more-year | Predicted: One-more-year
Test 7: True Song: Grimmer | Predicted: Grimmer
Test 8: True Song: Posthumous-forgivness | Predicted: Posthumous-forgivness
Test 9: True Song: Breathe-deeper | Predicted: Breathe-deeper
Test 10: True Song: One-more-year | Predicted: One-more-year

FINAL ACCURACY: 86.03% (234/272 correct)


In [22]:

def shazam_this(audio_file_path, model, database_matrix, database_labels, int_to_string_map, device):
    """
    Takes a raw audio file, processes it, and finds the closest song match.
    """
    print(f"Listening to: {audio_file_path}...")
    
    # 1. Load and force to Mono
    waveform, sample_rate = torchaudio.load(audio_file_path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        
    # 2. Resample if the user's phone didn't record at 16kHz
    if sample_rate != 16000:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    # 3. Convert to Mel Spectrogram
    mel_transform = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=512, n_mels=64)
    amp_to_db = T.AmplitudeToDB()
    db_mel_spec = amp_to_db(mel_transform(waveform))

    # 4. Crop or Pad to exactly 157 frames (5 seconds)
    target_frames = 157
    current_frames = db_mel_spec.shape[2]
    
    if current_frames > target_frames:
        db_mel_spec = db_mel_spec[:, :, :target_frames]
    elif current_frames < target_frames:
        padding_amount = target_frames - current_frames
        db_mel_spec = F.pad(db_mel_spec, (0, padding_amount))

    # 5. Add a Batch dimension (Shape becomes [1, 1, 64, 157]) and move to GPU
    db_mel_spec = db_mel_spec.unsqueeze(0).to(device)

    # 6. Extract the 128-number coordinate and search!
    with torch.no_grad():
        embedding = model(db_mel_spec)
        distances = torch.cdist(embedding.cpu(), database_matrix)
        closest_index = torch.argmin(distances).item()
        
    # 7. Decode the result
    predicted_song = int_to_string_map[database_labels[closest_index]]
    print(f"🎵 MATCH FOUND: {predicted_song}")
    
    return predicted_song

In [ ]:
shazam_this('pevanje.wav',model,database_matrix)